## this notebook is used to run GWAS on the "Cohort Builder"-derived cohort

adapted from notebook:

"5 - CB GWAS MF"

in workspace "Hypothyroidism genomics v7"


In [ ]:
# !pip install polars

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import json
import re
from google.cloud import bigquery
import polars as pl
#for gwas
import logging
import matplotlib.pyplot as plt
import pyspark
import hail as hl
from tqdm.auto import tqdm
%matplotlib inline

In [ ]:
def wb(*args):
    """Run a wb command and return parsed JSON."""
    cmd = ["wb", *args, "--format=json"]
    result = subprocess.check_output(cmd, text=True)
    return json.loads(result)


# Get workspace info
workspace = wb("workspace", "describe")
google_project_id = workspace["googleProjectId"]

# Get resources
resources = wb("resource", "list")

# WORKSPACE_BUCKET
bucket_resources = [
    r for r in resources
    if r.get("resourceType") == "GCS_BUCKET"
    and "practical_considerations_bucket" in r.get("id", "")
    and "temporary" not in r.get("id", "")
]

if not bucket_resources:
    raise ValueError("No matching bucket found")

bucket = f"gs://{bucket_resources[0]['bucketName']}"

# WORKSPACE_CDR
bq_resources = [
    r for r in resources
    if r.get("resourceType") in {"BQ_DATASET", "BIGQUERY_DATASET"}
]

cdr_resources = [
    r for r in bq_resources
    if re.match(r"^C\d{4}Q\d+R\d+$", r.get("datasetId", ""))
]

if not cdr_resources:
    raise ValueError("No matching CDR dataset found")

CDR = (
    f"{cdr_resources[0]['projectId']}."
    f"{cdr_resources[0]['datasetId']}"
)


In [ ]:
#initialize (only run once)
hl.init(gcs_requester_pays_configuration=google_project_id)

In [ ]:
logging.basicConfig(level='INFO')

In [ ]:
#read in acaf hail mt
# update to v9
# srWGS_snpindel_bucket = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel'
srWGS_snpindel_bucket = 'gs://vwb-aou-datasets-controlled/v7/wgs/short_read/snpindel'
wgs_path = f'{srWGS_snpindel_bucket}/acaf_threshold_v7.1/splitMT/hail.mt'
wgs = hl.read_matrix_table(wgs_path)

In [ ]:
#load in ancestry file
ancestry_pred_path = f'{srWGS_snpindel_bucket}/aux/ancestry/ancestry_preds.tsv'
ancestry_pred = hl.import_table(ancestry_pred_path,
                               key="research_id", 
                               impute=True, 
                               types={"research_id":"tstr","pca_features":hl.tarray(hl.tfloat)})

In [ ]:
# Define paths/filenames
pheno_path = f'{bucket}/hypothyroid_data/huan_phenotype_v4_covars.csv'
names = ['transancestry', 'eur', 'noneur', 'afr', 'amr']
sex = ['a','f','m']

pheno_path2 = f'{bucket}/hypothyroid_data/cb_v2_phenotype_covars.tsv'

In [ ]:
# Annotate wgs with ancestry
wgs = wgs.annotate_cols(ancestry_pred = ancestry_pred[wgs.s])

In [ ]:
class gwas:
    def __init__(self, path, sex):
        self.path = path
        self.sex = sex
        
        logging.info('Loading phenotype file...')

        self.phenotypes = hl.import_table(path,
                                types = {'person_id':hl.tstr,
                                        'age_at_cdr':hl.tfloat64,
                                        'sex_at_birth_Female':hl.tfloat64,
                                        'Hypothyroidism':hl.tfloat64,
                                        'hypothyroidism_eur':hl.tfloat64,
                                        'hypothyroidism_noneur':hl.tfloat64,
                                        'hypothyroidism_afr':hl.tfloat64
#                                         'hypothyroidisim_amr':hl.tfloat64
                                        },
                                impute = True,
                                 missing = ['NA', ''],
                                key = 'person_id')
        
        logging.info('Phenotype file read. Filtering...')
        if self.sex == 'm':
            self.phenotypes = self.phenotypes.filter(self.phenotypes.sex_at_birth_Female == 0)
        elif self.sex == 'f':
            self.phenotypes = self.phenotypes.filter(self.phenotypes.sex_at_birth_Female == 1)
            
        logging.info('Filtering complete. Annotating WGS...')

    
        self.wgs_annotated = wgs.annotate_cols(pheno = self.phenotypes[wgs.s])
        logging.info('Annotation complete.')

    def run_gwas(self, ancestry):
        self.ancestry = ancestry
        self.cov = [1.0, self.wgs_annotated.pheno.age_at_cdr, 
                    self.wgs_annotated.pheno.sex_at_birth_Female,
             self.wgs_annotated.ancestry_pred.pca_features[0], 
             self.wgs_annotated.ancestry_pred.pca_features[1], 
             self.wgs_annotated.ancestry_pred.pca_features[2],
             self.wgs_annotated.ancestry_pred.pca_features[3], 
             self.wgs_annotated.ancestry_pred.pca_features[4], 
             self.wgs_annotated.ancestry_pred.pca_features[5],
             self.wgs_annotated.ancestry_pred.pca_features[6],
             self.wgs_annotated.ancestry_pred.pca_features[7], 
             self.wgs_annotated.ancestry_pred.pca_features[8],
             self.wgs_annotated.ancestry_pred.pca_features[9], 
             self.wgs_annotated.ancestry_pred.pca_features[10], 
             self.wgs_annotated.ancestry_pred.pca_features[11],
             self.wgs_annotated.ancestry_pred.pca_features[12]]
        
        if self.ancestry == 'transancestry':
            y = self.wgs_annotated.pheno.Hypothyroidism
        elif self.ancestry == 'eur':
            y = self.wgs_annotated.pheno.hypothyroidism_eur
        elif self.ancestry == 'noneur':
            y = self.wgs_annotated.pheno.hypothyroidism_noneur
        elif self.ancestry == 'afr':
            y = self.wgs_annotated.pheno.hypothyroidism_afr
        elif self.ancestry == 'amr':
            y = self.wgs_annotated.pheno.hypothyroidism_amr
            
        logging.info('Beginning logistic regression...')
        self.log_reg = hl.logistic_regression_rows(
            test='wald',
            y=y,
            x=self.wgs_annotated.GT.n_alt_alleles(),
            covariates=self.cov
    #         pass_through = [wgs.rsid,wgs.variant_qc.AF]
        )
        logging.info('Logistic regression complete, flattening...')
        self.log_reg = self.log_reg.flatten()
        logging.info('Complete')
        return self.log_reg


In [ ]:
class gwas2:
    def __init__(self, path, sex):
        self.path = path
        self.sex = sex
        
        logging.info('Loading phenotype file...')

        self.phenotypes = hl.read_table(path)
        
        logging.info('Phenotype file read. Filtering...')
        if self.sex == 'm':
            self.phenotypes = self.phenotypes.filter(self.phenotypes.sex_at_birth_Female == 0)
        elif self.sex == 'f':
            self.phenotypes = self.phenotypes.filter(self.phenotypes.sex_at_birth_Female == 1)
            
        logging.info('Filtering complete. Annotating WGS...')

    
        self.wgs_annotated = wgs.annotate_cols(pheno = self.phenotypes[wgs.s])
        logging.info('Annotation complete.')

    def run_gwas(self, ancestry):
        self.ancestry = ancestry
        self.cov = [1.0, self.wgs_annotated.pheno.age_at_cdr, 
                    self.wgs_annotated.pheno.sex_at_birth_Female,
             self.wgs_annotated.ancestry_pred.pca_features[0], 
             self.wgs_annotated.ancestry_pred.pca_features[1], 
             self.wgs_annotated.ancestry_pred.pca_features[2],
             self.wgs_annotated.ancestry_pred.pca_features[3], 
             self.wgs_annotated.ancestry_pred.pca_features[4], 
             self.wgs_annotated.ancestry_pred.pca_features[5],
             self.wgs_annotated.ancestry_pred.pca_features[6],
             self.wgs_annotated.ancestry_pred.pca_features[7], 
             self.wgs_annotated.ancestry_pred.pca_features[8],
             self.wgs_annotated.ancestry_pred.pca_features[9], 
             self.wgs_annotated.ancestry_pred.pca_features[10], 
             self.wgs_annotated.ancestry_pred.pca_features[11],
             self.wgs_annotated.ancestry_pred.pca_features[12]]
        
        if (self.sex == 'f') or (self.sex == 'm'):
            self.cov = [1.0, self.wgs_annotated.pheno.age_at_cdr,
             self.wgs_annotated.ancestry_pred.pca_features[0], 
             self.wgs_annotated.ancestry_pred.pca_features[1], 
             self.wgs_annotated.ancestry_pred.pca_features[2],
             self.wgs_annotated.ancestry_pred.pca_features[3], 
             self.wgs_annotated.ancestry_pred.pca_features[4], 
             self.wgs_annotated.ancestry_pred.pca_features[5],
             self.wgs_annotated.ancestry_pred.pca_features[6],
             self.wgs_annotated.ancestry_pred.pca_features[7], 
             self.wgs_annotated.ancestry_pred.pca_features[8],
             self.wgs_annotated.ancestry_pred.pca_features[9], 
             self.wgs_annotated.ancestry_pred.pca_features[10], 
             self.wgs_annotated.ancestry_pred.pca_features[11],
             self.wgs_annotated.ancestry_pred.pca_features[12]]
        
        if self.ancestry == 'transancestry':
            y = self.wgs_annotated.pheno.Hypothyroidism
        elif self.ancestry == 'eur':
            y = self.wgs_annotated.pheno.hypothyroidism_eur
        elif self.ancestry == 'noneur':
            y = self.wgs_annotated.pheno.hypothyroidism_noneur
        elif self.ancestry == 'afr':
            y = self.wgs_annotated.pheno.hypothyroidism_afr
        elif self.ancestry == 'amr':
            y = self.wgs_annotated.pheno.hypothyroidism_amr
        elif self.ancestry == 'eas':
            y = self.wgs_annotated.pheno.hypothyroidism_eas
        elif self.ancestry == 'sas':
            y = self.wgs_annotated.pheno.hypothyroidism_sas
        

            
        logging.info('Beginning logistic regression...')
        self.log_reg = hl.logistic_regression_rows(
            test='wald',
            y=y,
            x=self.wgs_annotated.GT.n_alt_alleles(),
            covariates=self.cov
    #         pass_through = [wgs.rsid,wgs.variant_qc.AF]
        )
        logging.info('Logistic regression complete, flattening...')
        self.log_reg = self.log_reg.flatten()
        logging.info('Complete')
        return self.log_reg


In [ ]:
test_runs = gwas(pheno_path2, 'all')

In [ ]:
test_runs.run_gwas('transancestry').export(f'{bucket}/GWAS/CB/hypo_transancestry_7_30_26.tsv.bgz')

# test_runs.run_gwas('eur').export(f'{bucket}/GWAS/CB/hypo_eur_5_1_23.tsv.bgz')

# test_runs.run_gwas('afr').export(f'{bucket}/GWAS/CB/hypo_afr_5_1_23.tsv.bgz')

# test_runs.run_gwas('amr').export(f'{bucket}/GWAS/CB/hypo_amr_5_1_23.tsv.bgz')

# test_runs.run_gwas('eas').export(f'{bucket}/GWAS/CB/hypo_eas_5_1_23.tsv.bgz')

# test_runs.run_gwas('sas').export(f'{bucket}/GWAS/CB/hypo_sas_5_1_23.tsv.bgz')

In [ ]:
!gsutil ls {bucket}/GWAS/